# Job Board Data Collection and Analysis (Python)

This notebook collects public job listings from two job-board APIs, then filters them by role, keywords, and experience phrases and exports CSV files.

**Why use these sources rather than scraping LinkedIn directly?**
- LinkedIn prohibits unauthorized scraping and may restrict accounts.
- These public job-board endpoints are intended for programmatic access, subject to each provider's terms.
- No web data source can guarantee every request will always succeed. This notebook validates responses, reports source-specific failures, and can still save data from the source that succeeds.

Sources:
- Arbeitnow public job-board API: https://www.arbeitnow.com/blog/job-board-api
- Remotive public remote-jobs API: https://remotive.com/remote-jobs/api

Remotive requires attribution and a link back to each listing, and advises against frequent polling. This notebook requests each feed once per run and preserves source URLs.

## 1. Install packages

Run this cell first. If you are using Command Prompt rather than a notebook, run `python -m pip install -U requests pandas beautifulsoup4`.

In [ ]:
%pip install -U requests pandas beautifulsoup4

## 2. Configure the search

Change the role keywords and location to match your interests. The APIs have different coverage: Arbeitnow is especially useful for European listings; Remotive focuses on remote jobs. Experience years are extracted from the text when explicitly mentioned; many job descriptions do not state a precise number.

In [ ]:
import re
import time
import html
from pathlib import Path
from datetime import datetime, timezone

import requests
import pandas as pd
from bs4 import BeautifulSoup

KEYWORDS = [
    "machine learning",
    "AI engineer",
    "data scientist",
    "Python",
    "knowledge graph",
    "RAG",
    "agentic AI",
    "ontology"
]

# Examples: "India", "Germany", "Remote", or "" to skip location filtering.
LOCATION_FILTER = ""

# Optional: "senior", "lead", "staff", "principal", "junior", "mid-level"
SENIORITY_FILTER = ""

# Set to None to retain all matching records.
MAX_ROWS_PER_SOURCE = None

OUTPUT_DIR = Path("job_board_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

USER_AGENT = "JobResearchNotebook/1.0 (personal research; respectful low-volume requests)"

print("Keywords:", KEYWORDS)
print("Location filter:", LOCATION_FILTER or "None")
print("Output directory:", OUTPUT_DIR.resolve())

## 3. Request helper and HTML cleanup

The helper uses timeouts and validates HTTP status codes and JSON. It makes one request per source, avoiding aggressive retry loops. A failed source is reported; it does not prevent the other source from running.

In [ ]:
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": USER_AGENT,
    "Accept": "application/json"
})

def get_json(url, params=None, timeout=25):
    response = SESSION.get(url, params=params, timeout=timeout)
    if not response.ok:
        snippet = response.text[:300].replace("\n", " ")
        raise RuntimeError(
            f"HTTP {response.status_code} from {response.url}. "
            f"Response preview: {snippet}"
        )
    try:
        data = response.json()
    except ValueError as exc:
        raise RuntimeError(
            f"Expected JSON from {response.url}, but response was not valid JSON."
        ) from exc
    if not isinstance(data, (dict, list)):
        raise RuntimeError("Unexpected JSON response type.")
    return data

def clean_html(value):
    if not value:
        return ""
    text = BeautifulSoup(str(value), "html.parser").get_text(" ", strip=True)
    return re.sub(r"\s+", " ", html.unescape(text)).strip()

def first_nonempty(*values):
    for value in values:
        if value is not None and value != "":
            return value
    return ""

## 4. Collect Arbeitnow listings

The free endpoint returns a page-based feed. This function follows pagination links where present, and stops safely if the API response shape changes. We preserve the original listing URL and source.

In [ ]:
ARBEITNOW_URL = "https://www.arbeitnow.com/api/job-board-api"

def fetch_arbeitnow(max_pages=10):
    rows = []
    next_url = ARBEITNOW_URL
    pages = 0

    while next_url and pages < max_pages:
        data = get_json(next_url)
        pages += 1

        if isinstance(data, dict):
            items = data.get("data", [])
            links = data.get("links", {})
            next_url = links.get("next") if isinstance(links, dict) else None
        elif isinstance(data, list):
            items = data
            next_url = None
        else:
            raise RuntimeError("Unexpected Arbeitnow response structure.")

        if not isinstance(items, list):
            raise RuntimeError("Arbeitnow 'data' field was not a list.")

        for item in items:
            if not isinstance(item, dict):
                continue
            rows.append({
                "source": "Arbeitnow",
                "job_id": first_nonempty(item.get("slug"), item.get("id")),
                "title": clean_html(item.get("title")),
                "company": clean_html(item.get("company_name")),
                "location": clean_html(item.get("location")),
                "description": clean_html(item.get("description")),
                "employment_type": clean_html(", ".join(item.get("job_types", []))
                                              if isinstance(item.get("job_types"), list)
                                              else item.get("job_types")),
                "tags": ", ".join(map(str, item.get("tags", [])))
                        if isinstance(item.get("tags"), list) else clean_html(item.get("tags")),
                "remote": item.get("remote", ""),
                "salary": first_nonempty(item.get("salary"), ""),
                "published_at": item.get("created_at", ""),
                "url": first_nonempty(item.get("url"), item.get("job_url")),
            })

        if not next_url:
            break
        time.sleep(1)

    return rows

arbeitnow_rows = []
try:
    arbeitnow_rows = fetch_arbeitnow()
    print(f"Arbeitnow: retrieved {len(arbeitnow_rows)} listings.")
except Exception as exc:
    print(f"Arbeitnow request failed: {type(exc).__name__}: {exc}")

## 5. Collect Remotive remote listings

Remotive's public API supports a search parameter. To keep the request volume low, this notebook fetches the feed once and performs keyword filtering locally. The listing URL is preserved for attribution.

In [ ]:
REMOTIVE_URL = "https://remotive.com/api/remote-jobs"

def fetch_remotive(limit=500):
    data = get_json(REMOTIVE_URL, params={"limit": limit})
    if not isinstance(data, dict):
        raise RuntimeError("Unexpected Remotive response structure.")

    items = data.get("jobs", [])
    if not isinstance(items, list):
        raise RuntimeError("Remotive 'jobs' field was not a list.")

    rows = []
    for item in items:
        if not isinstance(item, dict):
            continue
        rows.append({
            "source": "Remotive",
            "job_id": item.get("id", ""),
            "title": clean_html(item.get("title")),
            "company": clean_html(item.get("company_name")),
            "location": clean_html(item.get("candidate_required_location")),
            "description": clean_html(item.get("description")),
            "employment_type": clean_html(item.get("job_type")),
            "tags": ", ".join(map(str, item.get("tags", [])))
                    if isinstance(item.get("tags"), list) else clean_html(item.get("tags")),
            "remote": True,
            "salary": clean_html(item.get("salary")),
            "published_at": item.get("publication_date", ""),
            "url": first_nonempty(item.get("url"), item.get("job_url")),
        })
    return rows

remotive_rows = []
try:
    remotive_rows = fetch_remotive()
    print(f"Remotive: retrieved {len(remotive_rows)} listings.")
except Exception as exc:
    print(f"Remotive request failed: {type(exc).__name__}: {exc}")

## 6. Normalize, deduplicate, and extract experience hints

The experience extractor only captures explicit phrases such as `5+ years`, `3 years of experience`, or `at least 4 years`. It does not guess when a listing does not state years of experience.

In [ ]:
all_rows = arbeitnow_rows + remotive_rows

COLUMNS = [
    "source", "job_id", "title", "company", "location", "description",
    "employment_type", "tags", "remote", "salary", "published_at", "url"
]

jobs = pd.DataFrame(all_rows)
if jobs.empty:
    jobs = pd.DataFrame(columns=COLUMNS)

for col in COLUMNS:
    if col not in jobs.columns:
        jobs[col] = ""

jobs = jobs[COLUMNS].copy()
jobs["title"] = jobs["title"].fillna("").astype(str)
jobs["description"] = jobs["description"].fillna("").astype(str)
jobs["company"] = jobs["company"].fillna("").astype(str)
jobs["location"] = jobs["location"].fillna("").astype(str)
jobs["tags"] = jobs["tags"].fillna("").astype(str)
jobs["url"] = jobs["url"].fillna("").astype(str)

jobs = jobs.drop_duplicates(subset=["source", "url"], keep="first")

def extract_experience_years(text):
    text = text.lower()
    patterns = [
        r"(\d{1,2})\s*\+\s*years?\s+(?:of\s+)?experience",
        r"(?:at least|minimum of|min\.)\s*(\d{1,2})\s*years?",
        r"(\d{1,2})\s*(?:-|to)\s*\d{1,2}\s*years?\s+(?:of\s+)?experience",
        r"(\d{1,2})\s*years?\s+(?:of\s+)?experience",
    ]
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return int(match.group(1))
    return pd.NA

jobs["experience_years_min"] = jobs.apply(
    lambda row: extract_experience_years(
        f"{row['title']} {row['description']}"
    ),
    axis=1
).astype("Int64")

jobs["collected_at_utc"] = datetime.now(timezone.utc).isoformat()

print("Total unique listings:", len(jobs))
display(jobs[["source", "title", "company", "location",
              "experience_years_min", "url"]].head(10))

## 7. Filter by keywords, location, and seniority

This is a local filter, so it does not send additional requests to the job boards. A job matches when any configured keyword appears in the title, description, or tags.

In [ ]:
def filter_jobs(df, keywords, location="", seniority=""):
    if df.empty:
        return df.copy()

    searchable = (
        df["title"].fillna("").astype(str) + " " +
        df["description"].fillna("").astype(str) + " " +
        df["tags"].fillna("").astype(str)
    ).str.lower()

    if keywords:
        keyword_pattern = "|".join(re.escape(k.lower()) for k in keywords)
        result = df[searchable.str.contains(keyword_pattern, regex=True, na=False)].copy()
    else:
        result = df.copy()

    if location.strip():
        result = result[
            result["location"].fillna("").astype(str).str.contains(
                re.escape(location.strip()), case=False, na=False
            )
        ]

    if seniority.strip():
        result = result[
            (
                result["title"].fillna("").astype(str) + " " +
                result["description"].fillna("").astype(str)
            ).str.contains(re.escape(seniority.strip()), case=False, na=False)
        ]

    return result

filtered_jobs = filter_jobs(
    jobs,
    KEYWORDS,
    location=LOCATION_FILTER,
    seniority=SENIORITY_FILTER
)

print(f"Matching listings: {len(filtered_jobs)} of {len(jobs)} collected.")
display(filtered_jobs[["source", "title", "company", "location",
                        "experience_years_min", "published_at", "url"]].head(20))

## 8. Export CSV files

The notebook always exports the collected dataset, even if no jobs match your chosen keywords. If both sources fail or return no listings, it writes a CSV with headers and prints a warning rather than pretending that data was collected.

In [ ]:
all_path = OUTPUT_DIR / "all_collected_jobs.csv"
filtered_path = OUTPUT_DIR / "filtered_jobs.csv"

jobs.to_csv(all_path, index=False, encoding="utf-8-sig")
filtered_jobs.to_csv(filtered_path, index=False, encoding="utf-8-sig")

print("Saved:", all_path.resolve(), f"({len(jobs)} rows)")
print("Saved:", filtered_path.resolve(), f"({len(filtered_jobs)} rows)")

if jobs.empty:
    print(
        "\nWARNING: No listings were collected from either source. "
        "Review the source-specific errors above, then try again later."
    )
elif filtered_jobs.empty:
    print(
        "\nNo listings matched your current filters. The source feeds worked; "
        "try broader keywords or clear LOCATION_FILTER / SENIORITY_FILTER."
    )

## 9. Inspect experience and role distribution

This summarizes only the listings collected in this run. It does not imply that every job board or every current vacancy is represented.

In [ ]:
if not filtered_jobs.empty:
    print("Listings by source:")
    display(filtered_jobs["source"].value_counts().rename("job_count").to_frame())

    print("Most common titles:")
    display(filtered_jobs["title"].value_counts().head(15).rename("job_count").to_frame())

    known_experience = filtered_jobs.dropna(subset=["experience_years_min"])
    print(
        "Listings with explicit experience-year text:",
        len(known_experience),
        "of",
        len(filtered_jobs)
    )
    if not known_experience.empty:
        display(
            known_experience["experience_years_min"]
            .value_counts()
            .sort_index()
            .rename("job_count")
            .to_frame()
        )
else:
    print("No matching listings to summarize.")

## Responsible use and troubleshooting

- LinkedIn's rules prohibit unauthorized scraping of its services. Use its authorized tools and access methods instead.
- Do not send requests repeatedly or in parallel. Remotive recommends low-frequency requests (a few times per day at most) and requires attribution and a link back to each listing.
- A `403` can mean access is denied; `429` means rate limiting; `5xx` usually indicates a temporary server issue.
- If one provider fails, the notebook still attempts the other provider.
- `experience_years_min` is a text-based heuristic, not a verified structured field. It will be blank when the listing does not clearly state years.
- Job postings may expire or change. Open the original `url` before applying.